In [10]:
import zipfile
with zipfile.ZipFile('/lila-viz.zip', 'r') as z:
    z.extractall('/content/')

print("Done! Files extracted.")

Done! Files extracted.


In [13]:
import os
import pyarrow.parquet as pq
import pandas as pd

base = "/content/lila-viz/player_data/February_10"
files = [f for f in os.listdir(base) if not f.startswith('.')]

print("Files found:", len(files))
print("First file:", files[0])

path = os.path.join(base, files[0])
df = pq.read_table(path).to_pandas()
df['event'] = df['event'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

print(df.head(5).to_string())
print("\nEvent types:", df['event'].unique())
print("Map:", df['map_id'].unique())
print("Shape:", df.shape)

Files found: 437
First file: b3340cc5-c9fa-4108-b56d-0728bc978a22_df85107f-d9e8-4c49-bb8e-e75e77ff53ab.nakama-0
                                user_id                                       match_id         map_id           x           y          z                      ts     event
0  b3340cc5-c9fa-4108-b56d-0728bc978a22  df85107f-d9e8-4c49-bb8e-e75e77ff53ab.nakama-0  AmbroseValley -315.939545  124.993996   4.074885 1970-01-21 11:52:07.710  Position
1  b3340cc5-c9fa-4108-b56d-0728bc978a22  df85107f-d9e8-4c49-bb8e-e75e77ff53ab.nakama-0  AmbroseValley -324.966980  125.126297  14.155012 1970-01-21 11:52:07.735  Position
2  b3340cc5-c9fa-4108-b56d-0728bc978a22  df85107f-d9e8-4c49-bb8e-e75e77ff53ab.nakama-0  AmbroseValley -303.042664  119.340866  23.113596 1970-01-21 11:52:07.740  Position
3  b3340cc5-c9fa-4108-b56d-0728bc978a22  df85107f-d9e8-4c49-bb8e-e75e77ff53ab.nakama-0  AmbroseValley -291.669037  115.060493  35.234829 1970-01-21 11:52:07.745  Position
4  b3340cc5-c9fa-4108-b56d-0728bc

In [15]:
import os
import pyarrow.parquet as pq
import pandas as pd

DATA_ROOT = "/content/lila-viz/player_data"
DAYS = ["February_10", "February_11", "February_12", "February_13",]

def load_all_data(data_root, days):
    frames = []
    for day in days:
        folder = os.path.join(data_root, day)
        for fname in os.listdir(folder):
            if fname.startswith('.'):
                continue
            fpath = os.path.join(folder, fname)
            try:
                df = pq.read_table(fpath).to_pandas()
                df['event'] = df['event'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
                # Detect human vs bot from filename
                user_id = fname.split('_')[0]
                df['is_bot'] = user_id.isdigit()
                df['day'] = day
                frames.append(df)
            except Exception as e:
                print(f"Skipped {fname}: {e}")
    return pd.concat(frames, ignore_index=True)

print("Loading all data... (may take 30-60 seconds)")
df_all = load_all_data(DATA_ROOT, DAYS)
print(f"✅ Done!")
print(f"\nTotal rows: {len(df_all):,}")
print(f"Unique players: {df_all['user_id'].nunique()}")
print(f"Unique matches: {df_all['match_id'].nunique()}")
print(f"Maps: {df_all['map_id'].unique()}")
print(f"Event types: {df_all['event'].unique()}")
print(f"Humans vs Bots: {df_all['is_bot'].value_counts().to_dict()}")
print(f"\nEvent breakdown:\n{df_all['event'].value_counts()}")

Loading all data... (may take 30-60 seconds)
✅ Done!

Total rows: 80,972
Unique players: 313
Unique matches: 712
Maps: ['AmbroseValley' 'Lockdown' 'GrandRift']
Event types: ['Position' 'Loot' 'BotPosition' 'BotKilled' 'BotKill' 'Kill' 'Killed'
 'KilledByStorm']
Humans vs Bots: {False: 59770, True: 21202}

Event breakdown:
event
Position         46310
BotPosition      20151
Loot             11649
BotKill           2186
BotKilled          639
KilledByStorm       31
Kill                 3
Killed               3
Name: count, dtype: int64


In [16]:
# Add pixel coordinates for minimap plotting
MAP_CONFIGS = {
    'AmbroseValley': {'scale': 900,  'origin_x': -370, 'origin_z': -473},
    'GrandRift':     {'scale': 581,  'origin_x': -290, 'origin_z': -290},
    'Lockdown':      {'scale': 1000, 'origin_x': -500, 'origin_z': -500},
}

def world_to_pixel(row):
    cfg = MAP_CONFIGS.get(row['map_id'])
    if cfg is None:
        return pd.Series([None, None])
    u = (row['x'] - cfg['origin_x']) / cfg['scale']
    v = (row['z'] - cfg['origin_z']) / cfg['scale']
    px = u * 1024
    py = (1 - v) * 1024
    return pd.Series([px, py])

print("Adding pixel coordinates...")
df_all[['px', 'py']] = df_all.apply(world_to_pixel, axis=1)

# Clamp to minimap bounds
df_all['px'] = df_all['px'].clip(0, 1024)
df_all['py'] = df_all['py'].clip(0, 1024)

# Save as CSV for fast reloading
df_all.to_csv('/content/lila_processed.csv', index=False)
print(f"✅ Saved! Sample coordinates:")
print(df_all[['map_id','x','z','px','py']].head(5).to_string())

# Quick sanity check - are coords in range?
print(f"\nPixel X range: {df_all['px'].min():.0f} → {df_all['px'].max():.0f}")
print(f"Pixel Y range: {df_all['py'].min():.0f} → {df_all['py'].max():.0f}")

Adding pixel coordinates...
✅ Saved! Sample coordinates:
          map_id           x          z         px          py
0  AmbroseValley -315.939545   4.074885  61.508785  481.194798
1  AmbroseValley -324.966980  14.155012  51.237569  469.725853
2  AmbroseValley -303.042664  23.113596  76.182569  459.532975
3  AmbroseValley -291.669037  35.234829  89.123229  445.741706
4  AmbroseValley -289.114197  42.443386  92.030069  437.539970

Pixel X range: 51 → 956
Pixel Y range: 75 → 918
